<a href="https://colab.research.google.com/github/i-ssa/PS-Agents/blob/modelling_approach_and_abstraction/search_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nairobi–Kisumu Search Tree

Bounded depth-first search over the directed graph from `graph.ipynb`. A town cannot repeat within one path, but can occur in separate branches. The visualisation limit does not establish the optimal route.

In [1]:
%pip install -q networkx matplotlib pydot
import json, networkx as nx, matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
Path("assets").mkdir(exist_ok=True)

## Load the graph
Upload `highway_graph.json` generated by `graph.ipynb` when prompted.

In [2]:
p=next((x for x in [Path("data/highway_graph.json"),Path("highway_graph.json")] if x.exists()),None)
if p is None:
    uploaded=files.upload()
    if "highway_graph.json" not in uploaded: raise FileNotFoundError("Upload highway_graph.json")
    p=Path("highway_graph.json")
with open(p) as f: G=nx.node_link_graph(json.load(f),directed=True)
START,GOAL="Nairobi CBD","Kisumu Town"
assert START in G and GOAL in G
print("States:",G.number_of_nodes(),"Edges:",G.number_of_edges(),"Reachable:",nx.has_path(G,START,GOAL))

KeyboardInterrupt: 

## Bounded DFS expansion
Every search-tree node represents a distinct path history. The same town may appear in multiple branches.

In [ ]:
MAX_DEPTH=8
MAX_NODES=250
tree=nx.DiGraph()
tree.add_node(0,town=START,depth=0,path=[START])
stack=[(0,START,[START])]
next_id=1
goals=[]
truncated=False
while stack:
    parent,town,path=stack.pop()
    if town==GOAL:
        goals.append(parent)
        continue
    if len(path)-1>=MAX_DEPTH: continue
    for neighbor in reversed(sorted(G.successors(town))):
        if neighbor in path: continue
        if tree.number_of_nodes()>=MAX_NODES:
            truncated=True
            break
        child=next_id; next_id+=1
        new_path=path+[neighbor]
        tree.add_node(child,town=neighbor,depth=len(path),path=new_path)
        tree.add_edge(parent,child)
        stack.append((child,neighbor,new_path))
    if truncated: break
print("Tree nodes:",len(tree),"Goals found:",len(goals),"Truncated:",truncated)
for i in goals[:10]: print(" → ".join(tree.nodes[i]["path"]))
if truncated: print("WARNING: Tree truncated; results are not exhaustive.")

## Visualise and export
Green nodes are goal states. This is a search-tree demonstration, not an optimality proof.

In [ ]:
from networkx.drawing.nx_pydot import graphviz_layout
import shutil
if shutil.which("dot") is None: raise RuntimeError("Install Graphviz: !apt-get install -y graphviz")
pos=graphviz_layout(tree,prog="dot")
labels=nx.get_node_attributes(tree,"town")
colors=["lightgreen" if i in goals else "lightblue" for i in tree]
plt.figure(figsize=(max(18,len(tree)*0.12),13))
nx.draw(tree,pos,labels=labels,with_labels=True,node_color=colors,node_size=1150,font_size=7,arrows=True,arrowsize=10)
plt.title("Bounded DFS Search Tree: Nairobi CBD to Kisumu Town")
plt.savefig("assets/search_tree.png",dpi=180,bbox_inches="tight")
plt.show()
with open("search_tree.json","w") as f: json.dump(nx.node_link_data(tree),f,indent=2)
print("Saved assets/search_tree.png and search_tree.json")

## GitHub submission
Include the exported image in `design.md` using `![Search tree](assets/search_tree.png)`. Commit the image and notebook to GitHub. For Question 3, calculate the shortest path on the **full weighted graph**, not on the truncated tree.